In [1]:
import pandas as pd
import numpy as np

students = pd.read_csv("../data/processed/students.csv")

courses = pd.read_csv("../data/processed/courses.csv")

interactions = pd.read_csv("../data/processed/student_interactions.csv")

print(students.head())
print(courses.head())
print(interactions.head())

   student_id skill_level preferred_subject learning_goal
0           1    Beginner           Science        Career
1           2    Beginner              Math          Exam
2           3    Beginner                AI        Career
3           4    Beginner           Science          Exam
4           5    Beginner           Science          Exam
   course_id course_title  subject    difficulty
0          1     Course 1       AI  Intermediate
1          2     Course 2  Science      Beginner
2          3     Course 3       AI      Advanced
3          4     Course 4     Math      Advanced
4          5     Course 5       AI      Beginner
   student_id  course_id  rating  completion_rate  quiz_score
0          80         40       1               67          56
1          91         12       3               90          62
2          73         10       1               58          31
3          62         43       5               72          95
4          75         34       1               6

In [2]:
data = interactions.merge(
    students,
    on='student_id'
)

data = data.merge(
    courses,
    on='course_id'
)

data.head()

,student_id,course_id,rating,completion_rate,quiz_score,skill_level,preferred_subject,learning_goal,course_title,subject,difficulty
0,80,40,1,67,56,Intermediate,AI,Career,Course 40,Programming,Advanced
1,91,12,3,90,62,Intermediate,Science,Skill Development,Course 12,Math,Beginner
2,73,10,1,58,31,Beginner,Science,Career,Course 10,Programming,Intermediate
3,62,43,5,72,95,Advanced,AI,Skill Development,Course 43,Programming,Advanced
4,75,34,1,60,83,Intermediate,Math,Skill Development,Course 34,Science,Beginner


In [3]:
data['performance_score'] = (
    data['quiz_score'] * 0.6 +
    data['completion_rate'] * 0.4
)

In [4]:
data['learning_need'] = 100 - data['performance_score']

In [5]:
courses['features'] = (
    courses['subject'] + ' ' +
    courses['difficulty']
)

courses.head()

,course_id,course_title,subject,difficulty,features
0,1,Course 1,AI,Intermediate,AI Intermediate
1,2,Course 2,Science,Beginner,Science Beginner
2,3,Course 3,AI,Advanced,AI Advanced
3,4,Course 4,Math,Advanced,Math Advanced
4,5,Course 5,AI,Beginner,AI Beginner


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(courses['features'])

cosine_sim = cosine_similarity(tfidf_matrix)

In [8]:
course_indices = pd.Series(
    courses.index,
    index=courses['course_title']
).drop_duplicates()

In [9]:
def hybrid_recommendation(student_id, top_n=5):

    student_data = data[data['student_id'] == student_id]

    preferred_subject = student_data[
        'preferred_subject'
    ].mode()[0]

    weak_performance = student_data[
        student_data['performance_score'] < 60
    ]

    if not weak_performance.empty:

        weak_subject = weak_performance[
            'subject'
        ].mode()[0]

    else:
        weak_subject = preferred_subject

    recommended_courses = courses[
        (courses['subject'] == weak_subject) &
        (courses['difficulty'] == 'Beginner')
    ]

    return recommended_courses[
        ['course_title', 'subject', 'difficulty']
    ].head(top_n)

In [10]:
hybrid_recommendation(student_id=5)

,course_title,subject,difficulty
4,Course 5,AI,Beginner
21,Course 22,AI,Beginner
23,Course 24,AI,Beginner
45,Course 46,AI,Beginner
47,Course 48,AI,Beginner


In [11]:
hybrid_recommendation(student_id=5)

,course_title,subject,difficulty
4,Course 5,AI,Beginner
21,Course 22,AI,Beginner
23,Course 24,AI,Beginner
45,Course 46,AI,Beginner
47,Course 48,AI,Beginner
